# 05 — Baseline and Ridge Modeling

This notebook turns the prepared stock data into a first model comparison. Work through the empty code cells in order. The intended outcome is a validation comparison of three Ridge settings, one selected fitted pipeline, and a final test evaluation.

## Theory before implementation

Imagine AAPL has strong recent momentum, low volatility, and unusually high volume. The baseline uses only its past 20-day return. Ridge learns how to combine four measurements into a score from historical examples. The question is whether this combination ranks future stock performance better on later dates.

For each stock and prediction date, the input vector contains trailing_return_5d, trailing_return_20d, volatility_20d, and relative_volume_20d. The target is target_percentile_5d: the percentile of that stock's subsequent five-observation return among stocks on the same date. A larger target means better future relative performance. It is an observed training outcome, not a probability.

A linear model predicts a score from a weighted sum:

$$
\hat y_i = b + \sum_{j=1}^{4} w_j z_{ij}.
$$

The intercept b sets the starting level, and each coefficient w describes how a standardized feature changes the predicted score while other features are held fixed. Scores can fall outside the target's 0–1 range; this experiment evaluates their ordering without clipping them.

StandardScaler transforms each feature using its training mean and standard deviation:

$$
z_{ij} = \frac{x_{ij}-\mu_j}{\sigma_j}.
$$

A value of 2 means two training standard deviations above the training mean. Scaling does not constrain values to a fixed range or make their distribution normal. It makes coefficient sizes comparable for the Ridge penalty. Reuse the training statistics on validation and test.

Ridge chooses its coefficients by minimizing:

$$
\sum_i (y_i-\hat y_i)^2 + \alpha\sum_j w_j^2.
$$

The first term rewards fitting the training targets. The second discourages large coefficients; the intercept is not penalized in our scikit-learn setup. Larger alpha means stronger shrinkage. This can stabilize overlapping signals such as 5-day and 20-day momentum, but improved future performance must be measured.

Use a scikit-learn Pipeline containing StandardScaler followed by Ridge. Calling fit on training inputs learns both scaling statistics and model weights. Calling predict on later inputs applies the fitted steps. Validation and test must not fit their own scalers.

Ridge fits squared error, while this project selects models using mean daily Spearman correlation. The two criteria measure different things. Spearman compares score and target ordering across stocks on each date. A useful fit to training targets does not guarantee a useful ranking on later dates.

Training learns weights, validation chooses alpha, and test evaluates the frozen choice. Five-day targets overlap, so keep the chronological splits and strict label-end exclusions from notebook 04. This experiment measures predictive rankings; portfolio returns, trading costs, and Sharpe ratio require the later backtest.

## Implementation

Keep the initial feature list, dates, and alpha candidates fixed throughout this comparison.

## 1. Import tools and load the full price history

Use pandas and NumPy for tables and checks. Import Pipeline, StandardScaler, and Ridge from scikit-learn. Reuse ExperimentConfig and EDA_DATA_PATH, build_features, build_targets, MODEL_FEATURE_COLUMNS, split_labeled_data, daily_spearman_ic, and summarize_daily_ic from the project modules.

Load the saved EDA CSV with date parsed as a datetime. Select the original market-data columns: date, symbol, open, high, low, close, adjusted_close, and volume. This supplies full historical prices without relying on derived columns from another notebook's live kernel.

In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from factor_model.features import build_features
from factor_model.metrics import daily_spearman_ic, summarize_daily_ic
from factor_model.models import ExperimentConfig
from factor_model.targets import build_targets
from factor_model.validation import MODEL_FEATURE_COLUMNS, split_labeled_data


## 2. Rebuild the labeled table with consistent column names

The earlier CSV exports use names such as five_day_return, while the current modules use trailing_return_5d. Rebuild features with build_features rather than mixing the two schemas.

Keep the full price history for target construction. Select eligible feature rows whose four MODEL_FEATURE_COLUMNS are finite, then call build_targets with the full prices, eligible features, and a horizon of five observations. This preserves the actual stock timeline when looking ahead. Confirm that baseline_score, target_percentile_5d, and label_end_date are present.

In [13]:
from factor_model.config import EDA_DATA_PATH

prices = pd.read_csv(EDA_DATA_PATH, parse_dates=["date"])
features = build_features(prices)

eligible_features = features.dropna(
    subset=list(MODEL_FEATURE_COLUMNS)
).copy()
df = build_targets(prices, eligible_features, 5)

## 3. Reproduce the chronological splits

Create an ExperimentConfig using the same ten symbols and the original period: start 2018-01-01, exclusive end 2025-01-01, validation start 2022-01-01, test start 2023-01-01, and target horizon 5. Use split_labeled_data to obtain splits.train, splits.valid, and splits.test.

Summarize row counts and prediction-date ranges. Verify that training label end dates are strictly before validation starts and validation label end dates are strictly before test starts. Keep all stocks from each date together. Preserve these split tables for every candidate.

In [14]:
from datetime import date

conf = ExperimentConfig(
    symbols=(
        "AAPL", "MSFT", "JPM", "JNJ", "XOM", "WMT", "HD", "CAT", "NEE", "AMGN"), 
    start_date=date(2018, 1, 1),
    end_date=date(2025, 1, 1),
    validation_start=date(2022, 1, 1),
    test_start=date(2023, 1, 1),
    target_horizon=5)

splits = split_labeled_data(df, conf, MODEL_FEATURE_COLUMNS)



## 4. Select model inputs, targets, and evaluation metadata

Use the four MODEL_FEATURE_COLUMNS in the same order for X_train and X_valid. Select target_percentile_5d for y_train and y_valid. Keep date, symbol, and label_end_date separately for evaluation.

Forward returns, target percentiles, and label_end_date contain future information and must not enter X. baseline_score is the comparison score; it duplicates trailing_return_20d and is not an additional model input. Check that X and y describe exactly the same rows before fitting.

In [18]:
X_train = splits.train[[*MODEL_FEATURE_COLUMNS]]
X_valid = splits.valid[[*MODEL_FEATURE_COLUMNS]]
X_test = splits.test[[*MODEL_FEATURE_COLUMNS]]

y_train = splits.train["target_percentile_5d"]
y_valid = splits.valid["target_percentile_5d"]
y_test = splits.test["target_percentile_5d"]

## 5. Establish validation baseline performance

Evaluate baseline_score against target_percentile_5d on splits.valid with daily_spearman_ic. Use summarize_daily_ic to report mean, median, fraction_positive, defined_day_count, and undefined_day_count.

Retain the daily Series as well as the summary. The positive fraction uses only dates with a defined correlation. Constant scores or targets produce an undefined correlation and must not be turned into zero. This baseline gives every Ridge candidate the same comparison on the same validation dates.

In [20]:
baseline_daily_ic = daily_spearman_ic(splits.valid)

baseline_summary = summarize_daily_ic(baseline_daily_ic)

baseline_summary

{'mean': -0.0037447647203744726,
 'median': -0.01818181818181818,
 'fraction_positive': 0.483739837398374,
 'defined_day_count': 246,
 'undefined_day_count': 0}

## 6. Fit one StandardScaler–Ridge pipeline

Build a Pipeline with StandardScaler followed by Ridge, starting with alpha=1.0. Fit it on X_train and y_train only, then predict scores for X_valid.

Inspect the fitted scaler's mean and scale and the Ridge coefficients. Associate each coefficient with its input column in the original order. The coefficients describe the fitted linear relationship in standardized feature units; they do not establish causes. Record this pipeline as a first candidate, not a final selection.

In [24]:
model = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
model.fit(X_train, y_train)
predictions = model.predict(X_valid)

## 7. Evaluate the first model on validation dates

Copy the validation metadata and observed target into a results table. Add the predictions as ridge_score, preserving their exact row alignment. Pass score_column="ridge_score" and the target column explicitly to daily_spearman_ic.

Compare its summary with the baseline. Plot the two daily IC Series or their rolling means using the same dates and window. Inspect missing correlations rather than silently removing inconvenient dates. Keep model inputs and fitted parameters unchanged during this comparison.

In [26]:
validation_results = splits.valid.copy()
validation_results["ridge_score"] = predictions

ridge_daily_ic = daily_spearman_ic(
    validation_results,
    score_column="ridge_score"
)
ridge_summary = summarize_daily_ic(ridge_daily_ic)
ridge_summary

{'mean': -0.02281350086228135,
 'median': 0.006060606060606061,
 'fraction_positive': 0.5121951219512195,
 'defined_day_count': 246,
 'undefined_day_count': 0}

## 8. Compare the planned alpha values

Repeat training and validation scoring for alpha values 0.1, 1.0, and 10.0. Create a fresh Pipeline for each candidate and fit each on the same training rows. Store each fitted pipeline, its daily validation IC, and its summary.

Assemble a comparison table with alpha and the validation metrics. Select the highest finite mean daily validation IC; if means tie, use the larger alpha as declared in notebook 04. A candidate with no defined validation correlations cannot be selected. Compare the winner with the baseline as well: a model beating other Ridge candidates may still fail to beat the simple baseline.

In [31]:
results = []
candidate_models = {}

for alpha in [0.1, 0.5, 1.0, 2, 5, 10.0]:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])
    candidate.fit(X_train, y_train)

    validation_results = splits.valid.copy()
    validation_results["ridge_score"] = candidate.predict(X_valid)

    daily_ic = daily_spearman_ic(
        validation_results,
        score_column="ridge_score",
    )
    summary = summarize_daily_ic(daily_ic)

    results.append({"alpha": alpha, **summary})
    candidate_models[alpha] = candidate

comparison = pd.DataFrame(results)
comparison


,alpha,mean,median,fraction_positive,defined_day_count,undefined_day_count
0,0.1,-0.022814,0.006061,0.512195,246,0
1,0.5,-0.022814,0.006061,0.512195,246,0
2,1.0,-0.022814,0.006061,0.512195,246,0
3,2.0,-0.022814,0.006061,0.512195,246,0
4,5.0,-0.022814,0.006061,0.512195,246,0
5,10.0,-0.022616,0.006061,0.512195,246,0


## 9. Freeze the choice and evaluate test once

Record the selected alpha and keep its existing training-fitted pipeline. For this first experiment, do not refit it on combined training and validation data.

Only after selection, prepare X_test using the same feature order and predict the test scores. Evaluate both the frozen Ridge model and baseline on splits.test with the same daily IC metric. Report the test results without changing alpha, features, or boundaries in response to them. A later revision is a new experiment and must be documented as such.

In [33]:
ranked_candidates = comparison.loc[
    np.isfinite(comparison["mean"])
].sort_values(
    ["mean", "alpha"],
    ascending=[False, False],
)

if ranked_candidates.empty:
    raise ValueError("No candidate has a valid validation score")

best_alpha = ranked_candidates.iloc[0]["alpha"]
best_model = candidate_models[best_alpha]
test_results = splits.test.copy()
test_results["ridge_score"] = best_model.predict(X_test)

test_summaries = {}

for name, score_column in [
    ("baseline", "baseline_score"),
    ("ridge", "ridge_score"),
]:
    daily_ic = daily_spearman_ic(
        test_results,
        score_column=score_column,
    )
    test_summaries[name] = summarize_daily_ic(daily_ic)

pd.DataFrame(test_summaries).T

,mean,median,fraction_positive,defined_day_count,undefined_day_count
baseline,0.055911,0.066667,0.581489,497.0,0.0
ridge,0.011548,0.006061,0.517103,497.0,0.0


## 10. Document findings and prepare reusable training code

### Experiment setup

This experiment compares a simple 20-day momentum baseline with a StandardScaler + Ridge pipeline on ten stocks: AAPL, MSFT, JPM, JNJ, XOM, WMT, HD, CAT, NEE, and AMGN. The model inputs are trailing_return_5d, trailing_return_20d, volatility_20d, and relative_volume_20d. The target is target_percentile_5d: the within-date percentile of the following five trading observations' adjusted-close return.

The historical period starts on 2018-01-01 and ends before 2025-01-01. Training prediction dates precede 2022-01-01, validation dates are in 2022, and test dates start on 2023-01-01. Warm-up rows and unavailable future outcomes are excluded; training and validation labels crossing the next split boundary are purged. Scaling and Ridge weights are fitted only on training rows.

### Validation and model selection

The baseline had mean daily Spearman correlation -0.003745, median -0.018182, and positive correlation on 48.37% of 246 validation dates. None of these dates had undefined correlation.

The tested Ridge alpha values were 0.1, 0.5, 1, 2, 5, and 10. Values from 0.1 to 5 produced approximately the same mean validation correlation (-0.022814); alpha=10 achieved -0.022616 and was selected by the highest finite mean validation correlation, with larger alpha preferred in an exact tie. Its median was 0.006061 and positive-date fraction was 51.22%, with 246 defined dates and zero undefined dates. The selected Ridge model was still worse than the baseline by mean validation correlation.

### Final test results

The already trained alpha=10 pipeline was evaluated without refitting or changing its settings based on test outcomes. Results below are rounded from the saved section 9 output.

| Metric | Baseline | Ridge (alpha=10) |
| --- | ---: | ---: |
| Mean daily Spearman correlation | 0.055911 | 0.011548 |
| Median daily Spearman correlation | 0.066667 | 0.006061 |
| Fraction of positive-correlation dates | 58.15% | 51.71% |
| Defined dates | 497 | 497 |
| Undefined dates | 0 | 0 |

### Conclusion and limitations

The 20-day momentum baseline outperformed the selected Ridge model on mean correlation, median correlation, and fraction of positive dates in the test period. Combining these four features with Ridge did not improve test ranking performance in this experiment. The baseline showed modest positive ranking agreement, while Ridge's average agreement was closer to zero. These results do not establish profitability or statistical significance. The positive-date fraction is not prediction accuracy or a profitable-trade percentage.

The five-day outcomes overlap across neighboring prediction dates, so daily correlations are not independent observations. The small fixed universe, company-selection effects, and previously inspected historical sample limit generalization. Trading costs and executable entry timing have not been evaluated. Keep this test result as a fixed record rather than tuning alpha again against it.

### Next implementation step

Extract pipeline creation and validation-based candidate selection into training.py, retaining the baseline comparison and existing metrics.py scoring. Keep plots and interpretation in the notebook. Record the selected pipeline's fitted coefficients alongside any saved model artifact when implementing experiment persistence.